## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

In [2]:
# For installing the libraries & downloading models from HF Hub
#!pip install huggingface_hub==0.23.2 pandas==1.5.3 tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.22 sentence-transformers==2.3.1 numpy==1.25.2 -q

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

/Users/ashokankarunanidhi/AI-ML/GitHub Libraries/HugginFace_LLMs/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
## Model configuration
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

In [5]:
llm = Llama(
    model_path=model_path,
    n_threads=2,  # CPU cores
    n_batch=512,  # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_gpu_layers=43,  # Change this value based on your model and your GPU VRAM pool.
    n_ctx=4096,  # Context window
    verbose=False,
)

llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h80           (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h96 

#### Response

In [6]:
def response(query,max_tokens,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [7]:
max_tokens = 128
print(response("What treatment options are available for managing hypertension?", max_tokens))



Hypertension, or high blood pressure, is a common condition that can increase the risk of various health problems, including heart disease, stroke, and kidney damage. The good news is that there are several treatment options available for managing hypertension, and the choice of treatment depends on the severity of the condition, underlying causes, and individual health factors. Here are some common treatment options for managing hypertension:

1. Lifestyle modifications: Making lifestyle modifications is often the first line of treatment for managing hypertension. This may include adopting a healthy diet rich in fruits, vegetables, whole


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [8]:
max_tokens = 128
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(response(user_input, max_tokens))



Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are the general steps for managing sepsis in a critical care unit:

1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, and lactic acidosis. It is essential to have a high index of suspicion for sepsis, especially in patients with known infections or risk factors.
2.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [9]:
max_tokens = 128
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(user_input, max_tokens))



Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may start as a mild discomfort that gradually worsens. The pain may be constant or come and go, and it may be accompanied by cramping or bloating.
2. Loss of appetite: People with appendic


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [10]:
max_tokens = 128
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(user_input, max_tokens))



Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles, leading to hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to a problem with the immune system.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss:

1. Corticosteroids: Corticosteroids are anti-inflammatory


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [11]:
max_tokens = 128
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(user_input, max_tokens))



A person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, is typically diagnosed with a traumatic brain injury (TBI). The treatment for a TBI depends on the severity and location of the injury, as well as the individual's overall health and age.

Immediate treatment for a TBI may include:

1. Emergency medical care: This may include surgery to remove hematomas or other obstructions, as well as treatment for other injuries that may have occurred at the same time as the TBI.
2. Med


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [12]:
max_tokens = 128
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(user_input, max_tokens))



First and foremost, if a person has fractured their leg during a hiking trip, it is essential to ensure their safety and prevent further injury. Here are some necessary precautions and treatment steps:

1. Assess the situation: Check the extent of the injury and assess the person's condition. If the fracture is open or the person is in severe pain, immobilize the leg with a splint or a makeshift sling to prevent any movement.
2. Call for help: If possible, call for emergency medical assistance. If there is no cell phone reception, try to


### **Observation**
- Since the max_token was 128, all the model response was in-complete once the taken limit was met. 
- Lets increase the max_tokens to get the complete response

## Question Answering using LLM with Prompt Engineering

In [13]:
system_prompt = "You are a medical assistant. You will be given a medical query and you have to answer it in a detailed manner. You can also provide the treatment options available for the query."

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [14]:
max_tokens = 512
user_input = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
print(response(user_input, max_tokens))


Sepsis is a life-threatening condition that occurs when an infection spreads throughout the body and triggers a severe inflammatory response. In a critical care unit, managing sepsis requires a multidisciplinary approach and prompt recognition and intervention to prevent progression to septic shock. Here is a general protocol for managing sepsis in a critical care unit:
1. Early recognition and assessment: Identify patients at risk of sepsis based on clinical signs and laboratory results. Use the Sequential Organ Failure Assessment (SOFA) score to assess organ dysfunction.
2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and prevent hypotension. Use crystalloids initially, and consider colloids or blood products if fluid resuscitation is not sufficient.
3. Antibiotic therapy: Start broad-spectrum antibiotics as soon as possible based on the suspected infection site and microbiological cultures. Adjust antibiotic therapy based on cul

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [15]:
# Ask for structured outputs in the form of JSON / Tables
max_tokens = 512
user_input = '''
What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

The output should be in the form of a JSON object with the following structure:
{
    "symptoms": [
        "symptom1",
        "symptom2",
        ...
    ],
    "treatment": {
        "medication": [
            "medication1",
            "medication2",
            ...
        ],
        "surgery": {
            "procedure": "procedure_name",
            "description": "description of the procedure"
        }
    }
}
'''
print(response(user_input, max_tokens))


{
    "symptoms": [
        "Abdominal pain, usually in the lower right side of the belly",
        "Loss of appetite",
        "Nausea and vomiting",
        "Fever",
        "Swelling in the abdomen",
        "Feeling sick or uneasy",
        "Inability to pass gas or have a bowel movement"
    ],
    "treatment": {
        "medication": [],
        "surgery": {
            "procedure": "Appendectomy",
            "description": "An appendectomy is a surgical procedure to remove the appendix. The appendix is a small pouch that extends from the large intestine. It is located in the lower right side of the abdomen. An appendectomy is usually performed as an emergency procedure to remove an inflamed appendix (appendicitis) to prevent it from bursting and causing peritonitis, a serious inflammation of the abdominal cavity."
        }
    }
}


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [16]:
max_tokens = 512
user_input = system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(user_input, max_tokens))



Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that causes hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to a problem with the immune system.

There are several effective treatments for addressing sudden patchy hair loss:

1. Corticosteroids: Corticosteroids are anti-inflammatory medications that can help reduce inflammation and suppress the immune system response that causes hair loss. They can be applied topically to the affected area or taken orally.
2. Immunotherapy: Immunotherapy involves the use of medications that stimulate the immune system to attack specific cells or proteins. In the case of alopecia areata, immunotherapy may involve the use of injections of a substance called diphenylcyclopropenone (DPCP) or squaric acid dibutyl ester (SADBE).
3. Minoxidil: Minoxidil is a medication that is applied topically to the s

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [17]:
max_tokens = 512
user_input = system_prompt+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(user_input, max_tokens))



A brain injury, also known as traumatic brain injury (TBI), can result from various causes such as motor vehicle accidents, sports injuries, falls, or violence. The treatment for a brain injury depends on the severity and location of the injury, as well as the extent of brain function impairment. Here are some common treatments for brain injuries:

1. Emergency care: The first priority in treating a brain injury is to ensure the person's airway is clear, they are breathing, and their heart is beating. If the person is unconscious or has a severe head injury, they may require emergency surgery to remove hematomas or decompress skull fractures.
2. Medications: Depending on the symptoms, the healthcare provider may prescribe medications to manage various conditions such as pain, swelling, seizures, or infections. For instance, corticosteroids may be used to reduce swelling, and anticonvulsants to prevent seizures.
3. Rehabilitation: Rehabilitation is an essential part of the treatment f

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [18]:
max_tokens = 512
user_input = system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(user_input, max_tokens))


A leg fracture, especially one sustained during a hiking trip, can be a serious injury that requires prompt medical attention. Here are the necessary precautions and treatment steps for a person who has fractured their leg:
1. Assess the severity of the injury: Check the leg for signs of swelling, bruising, deformity, or open wounds. If there is significant swelling, bleeding, or the person is unable to bear weight on the leg, it is essential to seek medical help immediately.
2. Provide first aid: If the fracture is not severe, and the person is able to bear some weight on the leg, provide first aid by immobilizing the leg using a splint or a sling. Ensure that the person is comfortable and keeps the leg elevated to reduce swelling.
3. Pain relief: Provide pain relief using over-the-counter pain medications such as acetaminophen or ibuprofen. If the pain is severe, the person may require prescription pain medication from a healthcare professional.
4. Transportation: If the person is u

### **Observations**
- With system prompt and max_tokens increased to 512, the model response is getting better.
- However, the response is still generic.

## Data Preparation for RAG

### Importing libraries

In [19]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

In [20]:
manual_pdf_path = "data/medical_diagnosis_manual.pdf"

In [21]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [22]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [23]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
ashokan.karunanidhi@gmail.com
J01DSQ8EKU
for personal use by ashokan.karunanidh
shing the contents in part or full is liable
Page Number : 2
ashokan.karunanidhi@gmail.com
J01DSQ8EKU
This file is meant for personal use by ashokan.karunanidhi@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .....................................................................................................................................................................

#### Checking the number of pages

In [24]:
len(manual) # Number of pages in the PDF

4114

### Data Chunking

In [25]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=100,
)

In [26]:
document_chunks = text_splitter.split_documents(manual)

In [27]:
len(document_chunks) # Number of chunks in the PDF

4714

In [28]:
document_chunks[0].page_content # First chunk of the PDF

'ashokan.karunanidhi@gmail.com\nJ01DSQ8EKU\nfor personal use by ashokan.karunanidh\nshing the contents in part or full is liable'

In [29]:
document_chunks[1].page_content # Second chunk of the PDF

'ashokan.karunanidhi@gmail.com\nJ01DSQ8EKU\nThis file is meant for personal use by ashokan.karunanidhi@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [30]:
document_chunks[2].page_content # Third chunk of the PDF

"Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [31]:
document_chunks[3].page_content # Fourth chunk of the PDF

'491\nChapter 44. Foot & Ankle Disorders    .....................................................................................................................................\n502\nChapter 45. Tumors of Bones & Joints    ...............................................................................................................................\n510\n5 - Ear, Nose, Throat & Dental Disorders    ..................................................................................................................\n510\nChapter 46. Approach to the Patient With Ear Problems    ...........................................................................................\n523\nChapter 47. Hearing Loss    .........................................................................................................................................................\n535\nChapter 48. Inner Ear Disorders    ...................................................................................................

### **Observations**
- As expected, the chucks have a overlap due to **chunk_size=100**

### Embedding

In [32]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

/var/folders/cp/x7cqhrb94454k14rqblbk1v40000gn/T/ipykernel_25195/3102610685.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


In [33]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [34]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [35]:
embedding_1,embedding_2

([-0.017095085233449936,
  -0.00897021684795618,
  -0.004513148218393326,
  -0.024305086582899094,
  -0.017007583752274513,
  -0.013475744053721428,
  -0.0018025769386440516,
  0.03062763810157776,
  0.028761370107531548,
  0.002886739792302251,
  0.021394453942775726,
  -3.4338612749706954e-05,
  0.02167952060699463,
  -0.029433369636535645,
  -0.00893345195800066,
  -0.0012389542534947395,
  -0.019824357703328133,
  -0.038566138595342636,
  -0.013257975690066814,
  0.003914758563041687,
  0.01642543449997902,
  0.017465393990278244,
  -0.08706174790859222,
  -0.035773005336523056,
  -0.005650111474096775,
  0.030215678736567497,
  0.01988857053220272,
  0.0018866159953176975,
  0.05306582525372505,
  0.04551675170660019,
  -0.013139861635863781,
  -0.009765294380486012,
  0.039643317461013794,
  -0.038228485733270645,
  -0.026996415108442307,
  -0.008407387882471085,
  0.04314391687512398,
  -0.035689301788806915,
  -0.020665805786848068,
  -0.03826716169714928,
  -0.0097684757784008

### **Observations**
The chosen model **"thenlper/gte-large"** have a vector demention of 1024.

### Vector Database

In [36]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [37]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir
)

In [38]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/var/folders/cp/x7cqhrb94454k14rqblbk1v40000gn/T/ipykernel_25195/2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [39]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [40]:
vectorstore.similarity_search("Medical treatments for hair loss ",k=3)

[Document(metadata={'keywords': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'subject': '', 'moddate': '2025-05-05T23:48:14+00:00', 'page': 858, 'format': 'PDF 1.7', 'total_pages': 4114, 'file_path': 'data/medical_diagnosis_manual.pdf', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'author': '', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': 'data/medical_diagnosis_manual.pdf', 'modDate': 'D:20250505234814Z'}, page_content='loss and can stimulate hair\n[Table 86-2. Interpreting Findings in Alopecia]\ngrowth. Efficacy is usually evident within 6 to 8 mo of treatment. Adverse effects include decreased libido,\nerectile and ejaculatory dysfunction, hypersensitivity reactions, gynecomastia, and myopathy. There may\nbe a decrease in prostate-specific antigen levels in older men, which should be taken into account when\nthat test is used for cancer screen

### **Observations**
- The basic similarity search retuned a top 3 match from the document.
- However, the result is not directly related to the hair loss medical treatments but the content that have some similarity with the query.

### Retriever

In [41]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [42]:
rel_docs = retriever.get_relevant_documents("Medical treatments for hair loss ")
rel_docs

/var/folders/cp/x7cqhrb94454k14rqblbk1v40000gn/T/ipykernel_25195/3757021908.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("Medical treatments for hair loss ")


[Document(metadata={'total_pages': 4114, 'modDate': 'D:20250505234814Z', 'subject': '', 'format': 'PDF 1.7', 'creationDate': 'D:20120615054440Z', 'source': 'data/medical_diagnosis_manual.pdf', 'page': 858, 'author': '', 'creator': 'Atop CHM to PDF Converter', 'trapped': '', 'moddate': '2025-05-05T23:48:14+00:00', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'file_path': 'data/medical_diagnosis_manual.pdf', 'keywords': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)'}, page_content='loss and can stimulate hair\n[Table 86-2. Interpreting Findings in Alopecia]\ngrowth. Efficacy is usually evident within 6 to 8 mo of treatment. Adverse effects include decreased libido,\nerectile and ejaculatory dysfunction, hypersensitivity reactions, gynecomastia, and myopathy. There may\nbe a decrease in prostate-specific antigen levels in older men, which should be taken into account when\nthat test is used for cancer screen

In [43]:
model_output = llm(
      "Medical treatments for hair loss",
      max_tokens=128,
      temperature=0,
    )

In [44]:
model_output

{'id': 'cmpl-a8f4c7d2-b15a-49e7-a2fb-3439a81e8b4d',
 'object': 'text_completion',
 'created': 1748195511,
 'model': '/Users/ashokankarunanidhi/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf',
 'choices': [{'text': " are not always effective for everyone. In some cases, people may choose to explore alternative methods to help slow down or even reverse the process of hair loss. One such method is scalp acupuncture.\n\nScalp acupuncture is a form of traditional Chinese medicine that involves the insertion of thin needles into specific points on the scalp. The practice is based on the belief that the body has energy pathways, or meridians, that can be influenced by the insertion of needles. By stimulating these points, it is thought that the body's energy flow can be improved, leading to",
   'index': 0,
   'logprobs': None,
   'finish_reason': 'length'}],
 'usage': {'prompt_tokens

### **Observations**
- With LLM, the model response is much better. However, it still lacks the completeness and needs some improvements. 

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [45]:
qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [46]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [47]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [48]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,top_k=20))

Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy based on suspected source and sensitivity patterns.
2. Culture and sensitivity tests to guide antibiotic selection.
3. Continuation of antibiotics for at least 5 days after shock resolves and evidence of infection subsides.
4. Drainage of abscesses and surgical excision of necrotic tissues.
5. Normalization of blood glucose to improve outcome.
6. Corticosteroid therapy with replacement


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [49]:
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input,top_k=20)

'###Answer\nThe common symptoms for appendicitis include acute abdominal pain, usually located in the right lower quadrant, loss of appetite, nausea, and fever. Appendicitis cannot be cured via medicine alone, and the standard treatment is surgical removal of the appendix, known as appendectomy.'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [50]:
user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input,top_k=20))

Based on the context, the condition being described is alopecia areata. The effective treatments or solutions for addressing sudden patchy hair loss, as mentioned in the context, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The possible causes behind sudden patchy hair loss, as mentioned in the context, include it being an autoimmune disorder affecting


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [51]:
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input,top_k=20))

Based on the context, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, include ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, surgery for patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas, maintaining adequate brain perfusion and oxygenation, preventing complications of altered sensorium, and rehabilitation. In the first few days after the injury,


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [52]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input,top_k=20))

Based on the context, the person who has fractured their leg during a hiking trip should follow these steps:

1. Seek medical care as soon as possible if there is an odor emanating from the cast or if a fever develops, which may indicate infection.
2. Maintain good hygiene to prevent infection.
3. Use a splint to immobilize the injury if it is stable and does not require bed rest. A splint allows the person to apply ice and move more, and does not contribute to compartment syndrome.
4. If bed rest is required


### **Observations**
- With system prompt and user prompt template, the model response is finally close to the context of the questions being asked. 
- As a next step, we will try to fine tue some of the parameters to get even much better response. 

### Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [53]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input, max_tokens=512))

Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy based on suspected source and sensitivity patterns.
2. Culture and sensitivity tests to guide antibiotic selection.
3. Continuation of antibiotics for at least 5 days after shock resolves and evidence of infection subsides.
4. Drainage of abscesses and surgical excision of necrotic tissues.
5. Normalization of blood glucose to improve outcome.
6. Corticosteroid therapy with replacement doses.
7. Consideration of other emerging therapies such as cooling for hyperthermia and early treatment of renal failure.

The context does not provide information on the use of activated protein C or monoclonal antibodies to manage sepsis in a critical care unit.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [54]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?" 
print(generate_rag_response(user_input, max_tokens=512, temperature=0.5))

Based on the context, the protocol for managing sepsis in a critical care unit involves the following steps:
1. Obtaining specimens of blood, body fluids, and wound sites for Gram stain and culture.
2. Starting prompt empiric antibiotic therapy immediately after suspecting sepsis.
   - One regimen includes gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin.
   - Another regimen includes ceftazidime plus a fluoroquinolone.
3. Adding vancomycin if resistant staphylococci or enterococci are suspected.
4. Draining abscesses and excising necrotic tissues.
5. Normalizing blood glucose levels between 80 to 110 mg/dL using a continuous IV insulin infusion.
6. Considering corticosteroid therapy with replacement doses.
7. Administering activated protein C (drotrecogin alfa) if the patient has a significant risk of death as defined by an APACHE II score of > 25.
8. Monitoring for bleeding as a complication of activated protein C therapy.
9. Considering other emergi

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [55]:
user_input_2 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?" 
print(generate_rag_response(user_input, max_tokens=512, temperature=0.5))

Based on the context, the protocol for managing sepsis in a critical care unit includes the following steps:
1. Prompt empiric antibiotic therapy: Antibiotics should be started immediately after suspecting sepsis. The selection of antibiotics requires an educated guess based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and of sensitivity patterns common to that specific inpatient unit, and previous culture results.
2. Parenteral antibiotics: Antibiotics should be given after specimens of blood, body fluids, and wound sites have been taken for Gram stain and culture.
3. Culture and sensitivity results: The antibiotic regimen should be changed accordingly once culture and sensitivity results are available.
4. Continuation of antibiotics: Antibiotics should be continued for at least 5 days after shock resolves and evidence of infection subsides.
5. Drainage of abscesses and surgical excision of necrotic tissues: The patient's condition will cont

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [56]:
user_input_2 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?" 
print(generate_rag_response(user_input, max_tokens=512))

Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy based on suspected source and sensitivity patterns.
2. Culture and sensitivity tests to guide antibiotic selection.
3. Continuation of antibiotics for at least 5 days after shock resolves and evidence of infection subsides.
4. Drainage of abscesses and surgical excision of necrotic tissues.
5. Normalization of blood glucose to improve outcome.
6. Corticosteroid therapy with replacement doses.
7. Consideration of other emerging therapies such as cooling for hyperthermia and early treatment of renal failure.

The context does not provide information on the use of activated protein C or monoclonal antibodies to manage sepsis in a critical care unit.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [57]:
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?" 
print(generate_rag_response(user_input, max_tokens=512)) 

Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Prompt empiric antibiotic therapy based on suspected source and sensitivity patterns.
2. Culture and sensitivity tests to guide antibiotic selection.
3. Continuation of antibiotics for at least 5 days after shock resolves and evidence of infection subsides.
4. Drainage of abscesses and surgical excision of necrotic tissues.
5. Normalization of blood glucose to improve outcome.
6. Corticosteroid therapy with replacement doses.
7. Consideration of other emerging therapies such as cooling for hyperthermia and early treatment of renal failure.

The context does not provide information on the use of activated protein C or monoclonal antibodies to manage sepsis in a critical care unit.


### **Observations**
- As expected, increasing the max_tokens have improved the completeness of the response.
- As temperature increase, the randomness appear to be increasing.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [58]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [59]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""

In [60]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [61]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [62]:
ground,rel = generate_ground_relevance_response(user_input = "What is the protocol for managing sepsis in a critical care unit?", max_tokens=512)
print(ground,end="\n\n")
print(rel)

 Steps to evaluate the answer:
1. Identify the information in the context related to managing sepsis in a critical care unit.
2. Compare the information in the context to the answer to ensure that the answer is derived only from the context.

Explanation:
The context provides detailed information about the care of critically ill patients in an ICU, including supportive care, patient monitoring and testing, and specific treatments for various conditions. Among these, the context discusses the importance of prompt empiric antibiotic therapy for sepsis, the use of different antibiotics based on the suspected source and causative organisms, and the need for normalization of blood glucose and corticosteroid therapy. The context also mentions the use of activated protein C (drotrecogin alfa) and other emerging therapies for severe sepsis and septic shock.

The answer summarizes the information in the context regarding the protocol for managing sepsis in a critical care unit, including prompt

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [63]:
ground,rel = generate_ground_relevance_response(user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?", max_tokens=512) 
print(ground,end="\n\n")
print(rel)

 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis and its treatment.
2. Check if the answer is derived only from the information in the context.

Explanation:
The answer mentions the common symptoms for appendicitis, which are acute abdominal pain, usually located in the lower right quadrant of the abdomen, loss of appetite, nausea, and vomiting. It also states that appendicitis cannot be cured via medicine alone and the standard treatment is surgical removal of the appendix through an open or laparoscopic appendectomy. The answer further mentions that antibiotics are administered before the surgery to prevent infection and may be continued after the surgery until the patient's temperature and white blood cell count have normalized.

All the information in the answer is directly taken from the context. The context mentions that the treatment for appendicitis is surgical removal and that antibiotics are given before and after the surgery. 

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [64]:
ground,rel = generate_ground_relevance_response(user_input = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?", max_tokens=512) 
print(ground,end="\n\n")
print(rel)

 Steps to evaluate the answer:
1. Identify the question and the specific information being asked for.
2. Read the context to understand the information provided about the causes and treatments for various types of hair loss, including alopecia areata.
3. Determine if the answer is derived only from the information presented in the context.

Explanation:
The question asks about the effective treatments or solutions for addressing sudden patchy hair loss, specifically alopecia areata, and what could be the possible causes behind it. The context provides detailed information about the causes and treatments for various types of hair loss, including alopecia areata. The answer directly quotes the context and provides a summary of the treatments mentioned for alopecia areata. Therefore, the answer is derived only from the information presented in the context.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, the answer would receive a score of 5. The 

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [65]:
ground,rel = generate_ground_relevance_response(user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?", max_tokens=512)
print(ground,end="\n\n")
print(rel)

 Steps to evaluate the answer:
1. Identify the information in the context related to the recommended treatments for a person with a brain injury.
2. Compare the information in the context with the answer to ensure that the answer is derived only from the context and not from any external sources.

Explanation:
The answer mentions the recommended treatments for a person with a brain injury, which includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and surgery if necessary. It also mentions the importance of maintaining adequate brain perfusion and oxygenation in the first few days after the injury and the need for rehabilitation. The answer also mentions treatments for increased intracranial pressure, such as pentobarbital coma, hypothermia, corticosteroids, and neuroprotective agents, and states that their efficacy is not guaranteed. The answer also mentions that seizures should be treated promptly.

All of this information is directl

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [66]:
ground,rel = generate_ground_relevance_response(user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?", max_tokens=512)
print(ground,end="\n\n")
print(rel)

 Steps to evaluate the answer:
1. Identify the necessary precautions and treatment steps for a person who has fractured their leg.
2. Check if the answer is derived only from the information presented in the context.

The answer adheres to the metric as it mentions all the necessary precautions and treatment steps for a person who has fractured their leg based on the information provided in the context. Therefore, the metric is followed completely.

Rating:
5 (The metric is followed completely)

Based on the evaluation

 Steps to evaluate the context as per the metric:
1. Identify the main aspects of the question: The question asks about the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, as well as what should be considered for their care and recovery.
2. Determine if the context addresses all and only the important aspects of the question: The context provides information on fractures,


## Actionable Insights and Business Recommendations

- Vector database creation time increases with the number of pages in the PDF document.
- Retrieval parameter **`k`** is critical as the answer can be spread across multiple contexts.
- **`chunk_overlap`** ensures coherence, especially when context spans across chunks.
- **`max_tokens`** depends on query complexity; higher values yield detailed responses, while simple queries result in concise outputs despite large token limits due to prompt design and zero **`temperature`**.
- Refine prompt design and temperature settings to control response length and creativity.
- Continuously adjust RAG parameters based on specific use cases for optimal performance.
- Prioritize groundedness and relevance in evaluations to ensure reliable and contextually accurate outputs.
- Establish a feedback loop to fine-tune parameters, improving performance for diverse query types.

<font size=6 color='blue'>Power Ahead</font>
___